# G01: ReAct + Calculator (Qwen3-4B), chuyên đề Mũ-Logarit, 3 dạng: D89, D84, D47

Notebook chạy **lần lượt 3 dạng** mà Qwen3-4B zero-shot chưa đạt 100%, nạp model **một lần**. Mỗi dạng chạy đủ
90 câu và ghi một file `<mã dạng>_react_calculator_full90.csv` trong `/kaggle/working`, cuối cùng ghi `summary_G01.csv`.

| Dạng | Nội dung | Zero-shot đúng | Số lệnh Calculator tối đa | Độ dài prompt (ký tự) |
|---|---|---|---|---|
| D89 | đếm số nguyên dương $y$ để $A(x-1)e^x=y(e^x+xy-Bx^2-C)$ có nghiệm $x$ trong một khoảng | 0/90 | 11 | 21,134 |
| D84 | đếm số nguyên $y$ để $27^{3x^2+xy}=(1+xy)27^{Cx}$ có nghiệm $x\in(\frac13;A)$ | 0/90 | 7 | 20,257 |
| D47 | đếm số nguyên $x$ để tồn tại $y$ thực thỏa $\log_a(x+y)=\log_b(x^2+y^2)$ | 3/90 | 8 | 20,133 |

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool `Calculator` (sympy, chính xác tuyệt đối, có bộ nhớ biến)
theo vòng lặp ReAct thật: `Thought → Action → Action Input → Observation → ...`

Prompt mỗi dạng gồm 5 phần giống bộ Số phức và dạng D79: `PART_TOOL` + `PART_KIENTHUC` (dùng chung) + `PART_HUONGGIAI`
(hướng giải theo lời giải mẫu của câu gốc) + `PART_FEWSHOT` (các `Observation` là kết quả chạy thật bằng chính backend) + `PART_NHIEMVU`.
Tham số sinh: `temperature=0.3, top_p=0.95, top_k=20, seed=42, presence_penalty=1.2, max_new_tokens=9216`, ép mở đầu `<think>\nThought:`.

## Trước khi chạy

1. Gắn Kaggle Dataset chứa `plan_solve_prompts_mulogarit_merged.json` (đã dùng cho zero-shot). Notebook tự dò đường dẫn bằng `glob`.
2. Bật GPU (khuyến nghị 2×T4), chọn **Save & Run All**.
3. Muốn thử nhanh vài câu: đặt `RUN_STT = [1, 2, 3]` và/hoặc `RUN_TYPES` ở ô cấu hình.


In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec torchaudio
import vllm; print('vLLM:', vllm.__version__)


In [ ]:
import os, re, json, time, glob
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

GROUP = 'G01'
RUN_TYPES = ['D89', 'D84', 'D47']      # the types of this notebook, in running order
DATA_FILENAME = 'plan_solve_prompts_mulogarit_merged.json'
# The dataset folder name under /kaggle/input differs between accounts, so
# search for the file instead of hardcoding a path.
_matches = glob.glob(f'/kaggle/input/**/{DATA_FILENAME}', recursive=True)
assert _matches, f'{DATA_FILENAME} not found under /kaggle/input. Add the dataset to the notebook first.'
DATA_PATH = _matches[0]

RUN_STT = None            # None = all 90 questions; for a quick test use e.g. [1, 2, 3]

MODEL = 'Qwen/Qwen3-4B'
# Same decoding settings as the Complex Numbers ReAct + Calculator runs.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_NEW_TOKENS = 9216

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    _all = json.load(f)
ALL_RECORDS = {ma: _all[ma] for ma in RUN_TYPES}
print({ma: len(v) for ma, v in ALL_RECORDS.items()})

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)


In [ ]:
# =====================================================================
# PROMPTS. PART_TOOL and PART_KIENTHUC are shared by every problem type of the
# chapter; PART_HUONGGIAI, PART_FEWSHOT and PART_NHIEMVU are specific to each type.
# TYPE_PROMPTS[type]['template'] is the five parts joined; {de_bai} is the question.
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: rate = log(6)/log(5)` → `Observation: rate = log(6)/log(5)`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: big = N(101**rate - 101, 20)`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Power is `**` (e.g. `x**2`, `101**rate`), not `^`. Square root is `sqrt(...)`; never the `√` symbol.
- Plain fractions are already exact: writing `7/2` gives exactly seven halves, never a decimal. You do NOT need any special function for fractions.
- Logarithms: `log(u)` is the NATURAL logarithm (write `log`, never `ln`). A logarithm to another base is written with the change of base formula, `log(u)/log(base)`, or equivalently `log(u, base)`. The exponential function is `exp(u)` and Euler's number is `E`.
- Absolute value is `Abs(u)`. Rounding: `floor(u)` is the greatest integer not above `u`, and `ceiling(u)` is the least integer not below `u`. `Min(u, v)` and `Max(u, v)` give the smaller and the larger of two numbers.
- Exact results stay exact (for example `log(8)/log(2)` becomes `3`). When you need a decimal value, for instance to see between which integers a number lies, write `N(expression, 20)`: this is the Calculator computing a decimal for you, not you approximating.
- To solve an equation, write `solve(expression, unknown)`, where the expression is the left side of the equation once everything is moved to one side so that it equals 0. It returns the list of solutions. A letter that the method tells you to use as an unknown (for example `x`, `y`, `t`, or the letters `a` and `b` of a formula) may be used inside `solve`, `diff`, `expand`, `factor` and `subs` without being assigned first; never store a number under that letter. Other tools: `expand(...)`, `factor(...)`, `diff(f, x)`, and `expr.subs(x, value)` to substitute.
- Some methods use these extra tools, always in exactly the form the method shows: `nsolve(expression, unknown, start)` finds a root numerically starting the search at `start`; `real_roots(polynomial)` lists the real roots of a polynomial exactly; `Sum(term, (k, first, last)).doit()` adds the terms exactly; `Count(condition, (n, low, high))` counts the integers `n` from `low` to `high` that make the condition true (it can be nested, and the counting letters need no earlier assignment); `Piecewise((value, condition), (other_value, True))` chooses a value by a condition; `And(...)` joins conditions; `oo` is infinity.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately. To compare two numbers by size, write `left < right`, `left <= right`, `left > right` or `left >= right`; the Calculator decides it with 50 digits of precision and answers `True` or `False`. Never write `==`.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation solving mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non real ones; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`) and discard it immediately if not. For a root confirmed real, use `re(root)`, not the raw value. Any comparison built on numeric roots should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root, in particular the final count and matching it against the answer options, stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''

PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF EXPONENTIALS AND LOGARITHMS**

1. **Powers**: for $a>0$ and real $m,n$: $a^m a^n=a^{m+n}$, $\dfrac{a^m}{a^n}=a^{m-n}$, $(a^m)^n=a^{mn}$, $(ab)^n=a^n b^n$, $a^{-n}=\dfrac{1}{a^n}$, $a^0=1$, and $a^{m/n}=\sqrt[n]{a^m}$.
2. **Exponential function** $y=a^x$ with $a>0$, $a\ne1$: it is defined for every real $x$, and $a^x>0$ always (it never equals 0 and is never negative). It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$.
3. **Exponential equations**: $a^{f}=a^{g}\iff f=g$. For a number $c$: $a^{f}=c$ has a solution only when $c>0$, and then $f=\log_a c$.
4. **Exponential inequalities** (same base): if $a>1$, then $a^{f}<a^{g}\iff f<g$ (direction kept). If $0<a<1$, then $a^{f}<a^{g}\iff f>g$ (direction reversed). In particular the sign of $a^{u}-a^{v}$ equals the sign of $u-v$ when $a>1$.
5. **Substitution $t=a^x$**: because $a^x>0$, the new variable must satisfy $t>0$. Then $a^{2x}=t^2$ and $a^{x+k}=a^k\,t$.
6. **Definition of the logarithm**: for $a>0$, $a\ne1$, $b>0$: $\log_a b=c\iff a^c=b$. The base must be positive and not equal to 1, and the argument must be strictly positive.
7. **Basic values**: $\log_a 1=0$, $\log_a a=1$, $\log_a a^c=c$, and $a^{\log_a b}=b$ for $b>0$.
8. **Product, quotient, power** (for $x,y>0$): $\log_a(xy)=\log_a x+\log_a y$, $\log_a\dfrac{x}{y}=\log_a x-\log_a y$, $\log_a x^{n}=n\log_a x$. For an even power of a number $x\ne0$ that may be negative: $\log_a x^{2k}=2k\log_a|x|$. Also $\log_{a^m}x=\dfrac{1}{m}\log_a x$ and $\log_a\sqrt[n]{x}=\dfrac{1}{n}\log_a x$.
9. **Change of base**: $\log_a b=\dfrac{\log_c b}{\log_c a}$ for any base $c>0$, $c\ne1$. Consequences: $\log_a b=\dfrac{1}{\log_b a}$ (for $b\ne1$) and $\log_a b\cdot\log_b c=\log_a c$.
10. **Natural and common logarithm**: $\ln x=\log_e x$ and $\log x=\log_{10}x$. In particular $\log_a b=\dfrac{\ln b}{\ln a}$.
11. **Exchanging base and exponent**: $a^{\log_b c}=c^{\log_b a}$ for $a,c>0$ and $b>0$, $b\ne1$. Reason: taking $\log_b$ of each side gives $\log_b c\cdot\log_b a$ on both sides, and two positive numbers with the same logarithm are equal.
12. **Logarithmic function** $y=\log_a x$: its domain is $x>0$. It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$. If $a>1$: $\log_a u<\log_a v\iff 0<u<v$ (direction kept). If $0<a<1$: $\log_a u<\log_a v\iff u>v>0$ (direction reversed). In both cases the arguments must be positive first.
13. **Logarithmic equations**: first state the domain (every argument $>0$). Then $\log_a f=\log_a g\iff f=g$, and $\log_a f=c\iff f=a^c$. A solution is accepted only if it satisfies the domain.
14. **Logarithmic inequalities with a constant**: if $a>1$: $\log_a f>c\iff f>a^c$ and $\log_a f<c\iff 0<f<a^c$. If $0<a<1$: $\log_a f>c\iff 0<f<a^c$ and $\log_a f<c\iff f>a^c$.
15. **Substitution $t=\log_a x$**: here $x>0$ and $t$ can be any real number. Then $\log_a^2 x=t^2$ and $x=a^t$.
16. **Always find the domain first**: every logarithm argument $>0$, every base positive and not 1, every denominator $\ne0$, every expression under a square root $\ge0$. Any value found later is accepted only if it satisfies this domain.
17. **Sign of a difference of two logarithms** (base $a>1$, arguments $u,v>0$): the sign of $\log_a u-\log_a v$ equals the sign of $u-v$. In particular $\log_a u-\log_a v=0\iff u=v$.
18. **Products and square roots**: $A\cdot B=0$ means $A=0$ or $B=0$. For $A\cdot\sqrt{B}=0$: either $A=0$ (and then $B\ge0$ must hold so that the square root exists), or $B=0$. The sign of a product $A\cdot B$ is decided by the signs of its two factors: $A\cdot B\le 0$ means one factor is $\ge0$ and the other is $\le0$.
19. **Derivatives**: $(a^x)'=a^x\ln a$, $(e^x)'=e^x$, $(a^u)'=u'\,a^u\ln a$. Also $(\log_a x)'=\dfrac{1}{x\ln a}$, $(\ln x)'=\dfrac{1}{x}$, $(\log_a u)'=\dfrac{u'}{u\ln a}$, and for a quotient $\left(\dfrac{u}{v}\right)'=\dfrac{u'v-uv'}{v^2}$.
20. **Derivative and monotonicity**: if $f'(x)>0$ on an interval, $f$ is strictly increasing there; if $f'(x)<0$, it is strictly decreasing. A strictly increasing function satisfies $f(u)<f(v)\iff u<v$ and $f(u)=f(v)\iff u=v$, so the equation $f=k$ has at most one solution.
21. **A power function with exponent above 1**: if $\alpha>1$ then $f(t)=t^{\alpha}-t$ has $f'(t)=\alpha t^{\alpha-1}-1$, and for $t\ge1$ we get $t^{\alpha-1}\ge1$, so $f'(t)\ge\alpha-1>0$: $f$ is strictly increasing on $[1,+\infty)$.
22. **Comparing the bases**: if $a>b>1$ then $\log_b a=\dfrac{\ln a}{\ln b}>1$.
23. **Integers**: a sum or difference of integers is an integer, and a positive integer is at least 1. Different integer values of $t$ give different values of $y=t-x$ for a fixed integer $x$, so counting the integers $y$ is the same as counting the integers $t$.
24. **Counting integers**: for real numbers $p<q$, the number of integers strictly between them (open interval $(p,q)$) is $(\lceil q\rceil-1)-(\lfloor p\rfloor+1)+1=\lceil q\rceil-\lfloor p\rfloor-1$. The number of integers in the closed interval $[p,q]$ is $\lfloor q\rfloor-\lceil p\rceil+1$. Here $\lfloor u\rfloor$ is the greatest integer not above $u$ and $\lceil u\rceil$ is the least integer not below $u$. The integers $1,2,\dots,T$ are exactly $T$ numbers.
25. **Quadratic equation** $ax^2+bx+c=0$ ($a\ne0$): $\Delta=b^2-4ac$. If $\Delta>0$ there are two distinct real roots $x_{1,2}=\dfrac{-b\pm\sqrt{\Delta}}{2a}$; if $\Delta=0$ one double root; if $\Delta<0$ no real root. Viète: $x_1+x_2=-\dfrac{b}{a}$ and $x_1x_2=\dfrac{c}{a}$.
26. **Quadratic inequality**: for $a>0$ and $\Delta>0$ with roots $x_1<x_2$: $ax^2+bx+c<0\iff x_1<x<x_2$, and $ax^2+bx+c>0\iff x<x_1$ or $x>x_2$. The two roots themselves are included only for the non strict signs $\le$ and $\ge$.
27. **Two distinct positive roots**: the quadratic $at^2+bt+c=0$ has two distinct positive roots if and only if $\Delta>0$, $x_1+x_2>0$ and $x_1x_2>0$.
28. **Squares and square roots**: $\sqrt{u^2}=|u|$. For $k>0$: $x^2<k\iff-\sqrt{k}<x<\sqrt{k}$ and $x^2>k\iff x<-\sqrt{k}$ or $x>\sqrt{k}$.
29. **Inequality of the arithmetic and geometric means**: for $u,v\ge0$: $u+v\ge2\sqrt{uv}$, with equality if and only if $u=v$.
30. **Bunyakovsky inequality** (also called Cauchy and Schwarz): for real numbers $p,q,x,y$: $(px+qy)^2\le(p^2+q^2)(x^2+y^2)$, with equality if and only if $(p,q)$ and $(x,y)$ are proportional.
31. **Values of a continuous monotone function**: if $f$ is continuous and strictly increasing on $[p,q]$, then on the open interval $(p,q)$ it takes every value strictly between $f(p)$ and $f(q)$ exactly once, and no other value. So $f(x)=k$ has a solution in $(p,q)$ if and only if $f(p)<k<f(q)$. For a strictly decreasing $f$ the roles of $f(p)$ and $f(q)$ swap.
32. **Behaviour at the ends** (base $a>1$): $a^x\to0^+$ when $x\to-\infty$ and $a^x\to+\infty$ when $x\to+\infty$; $\ln x\to-\infty$ when $x\to0^+$ and $\ln x\to+\infty$ when $x\to+\infty$. A function that goes from $-\infty$ up to a local maximum, down to a local minimum, then up to $+\infty$ crosses the $x$ axis 3 times if the maximum is positive and the minimum is negative, and once otherwise.
33. **Quadratic in $t=a^x>0$**: the roots of $t^2+Bt+D=0$ are accepted only if they are positive. If $D<0$ the roots have opposite signs, and $t^2+Bt+D>0$ with $t>0$ means $t$ above the larger root. A negative sign in front of a number is part of the number: for example in $t^2-12t+35$ we have $B=-12$.
'''

TYPE_PROMPTS = {}

# ---------------------------------------------------------------- D89
D89_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: the problem gives the equation $A(x-1)e^x=y\,(e^x+xy-Bx^2-C)$ and an open interval $(x_1;x_2)$ with $x_1\ge1$, and asks how many positive integers $y$ make the equation have a real solution $x$ in that interval. (If the coefficient of $x^2$ is not written it is 1, and if there is no constant term then $C=0$.)

**The function of $x$ for a fixed $y$ (as in the original solution).** The equation is equivalent to $f(x)=0$ with
$$f(x)=A(x-1)e^x-y\,(e^x+xy-Bx^2-C),\qquad f'(x)=Axe^x-y\,e^x-y^2+2Bxy .$$
(In the original problem $A=4$, $B=2$ and the derivative factors as $(e^x+y)(4x-y)$, with the single critical point $x=\frac y4$; the original solution then studies the position of $\frac y4$ with respect to the interval, and in every case the existence of a root comes down to the signs of $f$ at the two ends of the interval: $f(x_1)<0$ and $f(x_2)>0$.)

**The conditions at the two ends are quadratic inequalities in $y$.** For $x$ equal to an end, $f(x)$ is a quadratic polynomial in $y$:
$$-f(x)=x\,y^2+\big(e^x-Bx^2-C\big)\,y-A(x-1)e^x .$$
Its coefficients are $a=x$, $b=e^x-Bx^2-C$ and $c=-A(x-1)e^x$. On the interval $x>1$, so $a>0$ and $c<0$; then $ac<0$, the discriminant $b^2-4ac$ is positive, and the two roots have opposite signs. So there is exactly one positive root,
$$y=\psi(x)=\frac{-b+\sqrt{b^2-4ac}}{2a},$$
and $-f(x)=x\,y^2+b\,y+c$ is negative for $0<y<\psi(x)$ and positive for $y>\psi(x)$. Hence $f(x_1)<0\iff y>\psi(x_1)$, and $f(x_2)>0\iff y<\psi(x_2)$. (In the original problem these are the inequalities $y>5-e\approx2{,}28$ and $5y^2+(e^5-53)y-16e^5<0$, that is $y<14{,}25$.)

**Why these two conditions are exactly right for every $y$.** For each $x$ of the interval the positive root $\psi(x)$ is the only positive $y$ with $f(x)=0$. By implicit differentiation of $F(x,y)=f(x)$, $\psi'(x)=F_x/\sqrt{b^2-4ac}$ with $F_x=f'(x)$, and for the numbers of this type $\psi'(x)>0$ on the whole interval: $\psi$ is continuous and strictly increasing. So the values of $\psi$ on the open interval $(x_1;x_2)$ fill exactly the open interval $\big(\psi(x_1);\psi(x_2)\big)$, and a positive integer $y$ gives a root precisely when $\psi(x_1)<y<\psi(x_2)$. This replaces the case analysis on the position of the critical point.

**Counting.** We compute the two end values $M=\psi(x_1)$ and $N=\psi(x_2)$ as decimals (they are not integers), and the answer is the number of integers strictly between them, $\lceil N\rceil-\lfloor M\rfloor-1$. This count is then compared with the four options.
'''

D89_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số nguyên dương $y$ để tồn tại số thực $x\in(6;9)$ thỏa mãn $7(x-1)e^x=y(e^x+xy-4x^2-9)$?

A. $22$.
B. $24$.
C. $23$.
D. $21$.

<think>
Thought: I read the problem. The number in front of $(x-1)e^x$ is 7, the coefficient of $x^2$ is 4, the constant term is 9, and the interval is $(6;9)$, so the lower end is 6 and the upper end is 9. Viewed as a quadratic in $y$, the middle coefficient is $b(x)=e^x-4x^2-9$. At the lower end $x=6$ I store it as `b_low`.
Action: Calculator
Action Input: b_low = exp(6) - 4*6**2 - 9
Observation: b_low = -153 + exp(6)

Thought: The constant coefficient of the quadratic is $c(x)=-7(x-1)e^x$. At $x=6$ I store it as `c_low`.
Action: Calculator
Action Input: c_low = -7*(6 - 1)*exp(6)
Observation: c_low = -35*exp(6)

Thought: The positive root of $xy^2+b\,y+c=0$ is $\dfrac{-b+\sqrt{b^2-4ac}}{2a}$ with $a=x=6$. I use the stored `b_low` and `c_low`, and I wrap the formula in N with 20 digits because I need a decimal value to see between which integers it lies. I store it as `y_low`.
Action: Calculator
Action Input: y_low = N((-b_low + sqrt(b_low**2 - 4*6*c_low))/(2*6), 20)
Observation: y_low = 31.940521543111157670

Thought: Now the upper end $x=9$: the middle coefficient $b(9)=e^9-4\cdot9^2-9$ is stored as `b_high`.
Action: Calculator
Action Input: b_high = exp(9) - 4*9**2 - 9
Observation: b_high = -333 + exp(9)

Thought: The constant coefficient at $x=9$ is $-7(9-1)e^9$; I store it as `c_high`.
Action: Calculator
Action Input: c_high = -7*(9 - 1)*exp(9)
Observation: c_high = -56*exp(9)

Thought: The positive root at the upper end, with $a=9$, as a decimal with 20 digits, is stored as `y_high`.
Action: Calculator
Action Input: y_high = N((-b_high + sqrt(b_high**2 - 4*9*c_high))/(2*9), 20)
Observation: y_high = 54.907878456815602287

Thought: The values of $y$ fill the open interval from `y_low` to `y_high`. The number of integers strictly between them is the ceiling of `y_high` minus the floor of `y_low` minus 1, and this is the result.
Action: Calculator
Action Input: answer = ceiling(y_high) - floor(y_low) - 1
Observation: answer = 23

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $22$, B: $24$, C: $23$, D: $21$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $22$.
Action: Calculator
Action Input: test_A = Eq(answer, 22)
Observation: test_A = False

Thought: Option B is $24$.
Action: Calculator
Action Input: test_B = Eq(answer, 24)
Observation: test_B = False

Thought: Option C is $23$.
Action: Calculator
Action Input: test_C = Eq(answer, 23)
Observation: test_C = True

Thought: Option D is $21$.
Action: Calculator
Action Input: test_D = Eq(answer, 21)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: $22$ gives False, B: $24$ gives False, C: $23$ gives True, D: $21$ gives False. Exactly one option matches, and it is C.

</think>
Final Answer: \boxed{C}
'''

D89_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the number $A$ in front of $(x-1)e^x$; the coefficient $B$ of $x^2$ inside the bracket (it is 1 if no number is written); the constant term $C$ inside the bracket (it is 0 if there is none); and the two ends $x_1<x_2$ of the interval.
[Step 2] Store `b_low = exp(<x1>) - <B>*<x1>**2 - <C>`, then `c_low = -<A>*(<x1> - 1)*exp(<x1>)`, writing the numbers from the problem.
[Step 3] Store `y_low = N((-b_low + sqrt(b_low**2 - 4*<x1>*c_low))/(2*<x1>), 20)`.
[Step 4] Do the same at the upper end: store `b_high = exp(<x2>) - <B>*<x2>**2 - <C>`, then `c_high = -<A>*(<x2> - 1)*exp(<x2>)`, then `y_high = N((-b_high + sqrt(b_high**2 - 4*<x2>*c_high))/(2*<x2>), 20)`.
[Step 5] Store `answer = ceiling(y_high) - floor(y_low) - 1`.
[Step 6] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D89'] = {
    'template': PART_TOOL + PART_KIENTHUC + D89_HUONGGIAI + D89_FEWSHOT + D89_NHIEMVU,
    'free_unknowns': ['x'],
    'max_tool_calls': 25,
}


# ---------------------------------------------------------------- D84
D84_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: the problem gives the equation $27^{3x^2+xy}=(1+xy)\,27^{Cx}$ (with $C=3A$) and asks how many integers $y$ make it have a solution $x$ in the open interval $\left(\frac13;A\right)$.

**Case 1: $y\le0$.** The factor $1+xy$ equals a power of 27, which is positive, so $1+xy>0$. Since $x>\frac13$ this forces $y>-3$, so only $y=0,-1,-2$ can work, and we check them directly.
- For $y=0$ the equation is $27^{3x^2}=27^{Cx}$, so $3x^2=Cx$, that is $3x(x-A)=0$; the solutions $x=0$ and $x=A$ are not in the open interval, so $y=0$ is rejected.
- For $y=-1$ and $y=-2$ the equation has a solution in $\left(\frac13;-\frac1y\right)$ (the function $27^{3x^2+(y-C)x}-(1+xy)$ is negative at $x=\frac13$ and tends to a positive number at the right end of that interval), so both values are accepted.
This case gives exactly 2 values of $y$.

**Case 2: $y\ge1$.** Take the logarithm to the base 27 of both sides: $3x^2+xy=\log_{27}(1+xy)+Cx$. Divide by $x>0$ and move everything to one side:
$$g(x)=3x-C-\frac{\log_{27}(1+xy)}{x}+y=0.$$
The derivative is $g'(x)=3+\dfrac{\ln(1+xy)}{x^2\ln27}-\dfrac{y}{x(1+xy)\ln27}>3-\dfrac{1}{3x^2\ln3}\ge3-\dfrac{3}{\ln3}>0$ on $\left[\frac13;A\right]$, so $g$ is strictly increasing there. An increasing continuous function has a zero in the open interval exactly when $g\left(\frac13\right)\cdot g(A)<0$. Using $\ln(1+u)<u$ for $u>0$ one shows $g(A)>0$, so the condition is $g\left(\frac13\right)<0$. Substituting $x=\frac13$:
$$g\left(\tfrac13\right)=y+1-C-\log_3\left(1+\tfrac y3\right)<0\iff y-\log_3\left(1+\tfrac y3\right)<C-1.$$

**Counting the integers $y\ge1$.** The left side $h(y)=y-\log_3\left(1+\frac y3\right)$ is strictly increasing in $y$ (because $h'(y)=1-\dfrac{1}{(y+3)\ln3}>0$). Hence the equation $h(y)=C-1$ has exactly one root $y_0$, and the inequality $h(y)<C-1$ holds exactly for $y<y_0$. The positive integers below $y_0$ are $1,2,\dots,\lceil y_0\rceil-1$. If $y_0$ happened to be an integer itself, that integer is excluded because the inequality is strict; subtracting a tiny number such as $10^{-9}$ before taking the ceiling handles this case correctly. The number of positive integers is therefore $\lceil y_0-10^{-9}\rceil-1$.

**Finding $y_0$ with the Calculator.** The equation $y-\log_3\left(1+\frac y3\right)=C-1$ has no simple closed form, so the Calculator solves it numerically: `nsolve(expression, y, start)` returns the root of `expression = 0` found by starting the search at the number `start` (use $C$ as the starting value). A logarithm to the base 3 is written `log(u)/log(3)`.

**Conclusion.** The total number of integers $y$ is that count plus the 2 values $y=-1,-2$ of Case 1. This total is compared with the four options.
'''

D84_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số nguyên $y$ tồn tại sao cho $x\in\left(\dfrac{1}{3};2\right)$ thỏa mãn $27^{3x^2+xy}=(1+xy)27^{6x}$?

A. $5$.
B. $8$.
C. $6$.
D. $7$.

<think>
Thought: I read the problem. The right side is $27^{6x}$, so $C=6$ (the interval is $\left(\frac13;2\right)$, consistent with $C=3A$). For $y\ge1$ the condition is $y-\log_3\left(1+\frac y3\right)<C-1$, and its left side is increasing, so I need the single root $y_0$ of $y-\log_3\left(1+\frac y3\right)=6-1$. The Calculator finds it numerically with nsolve, starting the search at $y=6$ (the value of $C$); the logarithm to the base 3 is written $\log(\cdot)/\log(3)$.
Action: Calculator
Action Input: y_limit = nsolve(y - log(1 + y/3)/log(3) - (6 - 1), y, 6)
Observation: y_limit = 6.00000000000000

Thought: The positive integers $y$ with $y<y_0$ are $1,\dots,\lceil y_0\rceil-1$. To be safe if $y_0$ were itself an integer, I subtract $10^{-9}$ before taking the ceiling; the number of positive integers is stored as `positive_count`.
Action: Calculator
Action Input: positive_count = ceiling(y_limit - 10**(-9)) - 1
Observation: positive_count = 5

Thought: Case 1 of the method gave exactly 2 more values, $y=-1$ and $y=-2$, so the total is `positive_count + 2`.
Action: Calculator
Action Input: answer = positive_count + 2
Observation: answer = 7

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $5$, B: $8$, C: $6$, D: $7$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $5$.
Action: Calculator
Action Input: test_A = Eq(answer, 5)
Observation: test_A = False

Thought: Option B is $8$.
Action: Calculator
Action Input: test_B = Eq(answer, 8)
Observation: test_B = False

Thought: Option C is $6$.
Action: Calculator
Action Input: test_C = Eq(answer, 6)
Observation: test_C = False

Thought: Option D is $7$.
Action: Calculator
Action Input: test_D = Eq(answer, 7)
Observation: test_D = True

Thought: Lining up what I copied with the stored results: A: $5$ gives False, B: $8$ gives False, C: $6$ gives False, D: $7$ gives True. Exactly one option matches, and it is D.

</think>
Final Answer: \boxed{D}
'''

D84_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the number $C$ in the exponent $27^{Cx}$ on the right side of the equation (the bound $A$ of the interval is not needed).
[Step 2] Store `y_limit = nsolve(y - log(1 + y/3)/log(3) - (<C> - 1), y, <C>)`, writing the number $C$ twice. This is the root $y_0$.
[Step 3] Store `positive_count = ceiling(y_limit - 10**(-9)) - 1`. This is the number of positive integers $y$.
[Step 4] Store `answer = positive_count + 2`, where 2 stands for the values $y=-1$ and $y=-2$ of Case 1.
[Step 5] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D84'] = {
    'template': PART_TOOL + PART_KIENTHUC + D84_HUONGGIAI + D84_FEWSHOT + D84_NHIEMVU,
    'free_unknowns': ['y'],
    'max_tool_calls': 25,
}


# ---------------------------------------------------------------- D47
D47_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: the problem gives $\log_a(x+y)=\log_b(x^2+y^2)$ with two different bases $a$ and $b$ (with $a^2>b$), and asks how many integers $x$ admit a real number $y$ satisfying it.

**A common value and a system.** Let $t=\log_a(x+y)=\log_b(x^2+y^2)$. Then $x+y=a^t$ and $x^2+y^2=b^t$. Write $r=\log_a b$, that is $b=a^r$; then $b^t=(a^t)^r$, so the system gives the single relation
$$x^2+y^2=(x+y)^r,\qquad x+y>0.$$

**Cauchy gives the range of $t$ and shows $r<2$.** The inequality $(x+y)^2\le2(x^2+y^2)$ gives $a^{2t}\le2\,b^t$, that is $(a^2/b)^t\le2$, so $t\le\log_{a^2/b}2$. Since $a^2>b$ we have $r=\log_a b<2$, and this is what the later steps need.

**The candidates for $x$ (as in the original solution).** From $x^2\le x^2+y^2=b^t\le b^{t_{\max}}$ with $t_{\max}=\log_{a^2/b}2$ we get $|x|\le\sqrt{b^{t_{\max}}}$. In the original problem ($a=3$, $b=4$) this is $x^2\le4^{\log_{9/4}2}\approx1{,}89$, so the only candidates are $x\in\{-1,0,1\}$, and each candidate is then tested: $x=0$ works with $y=1$ ($t=0$), $x=1$ works with $y=0$ ($t=0$), and $x=-1$ is impossible. When $r=\log_a b$ is close to 2 this bound allows many candidates (for the bases $2^m$ and $2^n$ up to dozens), so instead of testing them one by one we find the exact largest admissible $x$ below.

**Negative $x$ is impossible.** Suppose $x\le-1$. Then $y=(x+y)-x\ge(x+y)+1$. If $x+y\ge1$, then $y^2>(x+y)^2\ge(x+y)^r$ (because $r<2$), so $x^2+y^2>(x+y)^r$, a contradiction. If $0<x+y<1$, then $y\ge1$, so $x^2+y^2\ge2>1>(x+y)^r$, again a contradiction. So $x\ge0$.

**$x=0$ and $x\ge1$.** For $x=0$ take $y=1$ (then $t=0$): it works. For $x>0$ put $y=\lambda x$; since $x+y>0$ we need $\lambda>-1$. The relation becomes $x^2(1+\lambda^2)=x^r(1+\lambda)^r$, that is
$$x=X(\lambda)=\left[\frac{(1+\lambda)^r}{1+\lambda^2}\right]^{\frac{1}{2-r}}.$$
The function $X$ is continuous on $(-1;+\infty)$, tends to 0 when $\lambda\to-1$ and when $\lambda\to+\infty$, and $X(0)=1$. Hence $x$ takes every value in $(0;X_{\max}]$, where $X_{\max}$ is the largest value of $X$.

**The largest value.** Set the derivative of $\ln X$ to zero: $\dfrac{r}{1+\lambda}=\dfrac{2\lambda}{1+\lambda^2}$, which simplifies to $(2-r)\lambda^2+2\lambda-r=0$. This quadratic has exactly one root greater than $-1$, namely
$$\lambda_0=\frac{-1+\sqrt{1+r(2-r)}}{2-r},$$
and $X_{\max}=X(\lambda_0)$.

**Counting.** The admissible integers are $x=0$ and $x=1,2,\dots,\lfloor X_{\max}\rfloor$ (note $X_{\max}\ge X(0)=1$). Their number is $\lfloor X_{\max}\rfloor+1$. This count is compared with the four options.
'''

D47_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số nguyên $x$ để tồn tại số thực $y$ thỏa mãn $\log_{128}(x+y)=\log_{4096}(x^2+y^2)$?

A. $6$.
B. $8$.
C. $7$.
D. $5$.

<think>
Thought: I read the problem. The base of the first logarithm is 128 and the base of the second is 4096. With $b=a^r$ the exponent is $r=\log_a b=\log_{128}4096$, which by the change of base formula is $\log(4096)/\log(128)$. I need it as a decimal for the next formulas, so I wrap it in N with 20 digits and store it as `ratio`.
Action: Calculator
Action Input: ratio = N(log(4096)/log(128), 20)
Observation: ratio = 1.7142857142857142857

Thought: The method gives $\lambda_0=\dfrac{-1+\sqrt{1+r(2-r)}}{2-r}$. I use the stored `ratio` for $r$, with 20 digits, and store the result as `lam`.
Action: Calculator
Action Input: lam = N((-1 + sqrt(1 + ratio*(2 - ratio)))/(2 - ratio), 20)
Observation: lam = 0.77200187265876558394

Thought: Now $X_{\max}=\left[\dfrac{(1+\lambda_0)^r}{1+\lambda_0^2}\right]^{1/(2-r)}$, with the stored `lam` and `ratio`, again with 20 digits. I store it as `x_max`.
Action: Calculator
Action Input: x_max = N(((1 + lam)**ratio/(1 + lam**2))**(1/(2 - ratio)), 20)
Observation: x_max = 6.0281276360344979165

Thought: The admissible integers are $0,1,\dots,\lfloor X_{\max}\rfloor$, which is the floor of `x_max` plus 1 numbers.
Action: Calculator
Action Input: answer = floor(x_max) + 1
Observation: answer = 7

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $6$, B: $8$, C: $7$, D: $5$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $6$.
Action: Calculator
Action Input: test_A = Eq(answer, 6)
Observation: test_A = False

Thought: Option B is $8$.
Action: Calculator
Action Input: test_B = Eq(answer, 8)
Observation: test_B = False

Thought: Option C is $7$.
Action: Calculator
Action Input: test_C = Eq(answer, 7)
Observation: test_C = True

Thought: Option D is $5$.
Action: Calculator
Action Input: test_D = Eq(answer, 5)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: $6$ gives False, B: $8$ gives False, C: $7$ gives True, D: $5$ gives False. Exactly one option matches, and it is C.

</think>
Final Answer: \boxed{C}
'''

D47_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the base $a$ of the first logarithm $\log_a(x+y)$ and the base $b$ of the second logarithm $\log_b(x^2+y^2)$.
[Step 2] Store `ratio = N(log(<b>)/log(<a>), 20)`, writing the two bases from the problem. This is $r=\log_a b$.
[Step 3] Store `lam = N((-1 + sqrt(1 + ratio*(2 - ratio)))/(2 - ratio), 20)`.
[Step 4] Store `x_max = N(((1 + lam)**ratio/(1 + lam**2))**(1/(2 - ratio)), 20)`.
[Step 5] Store `answer = floor(x_max) + 1`.
[Step 6] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D47'] = {
    'template': PART_TOOL + PART_KIENTHUC + D47_HUONGGIAI + D47_FEWSHOT + D47_NHIEMVU,
    'free_unknowns': [],
    'max_tool_calls': 25,
}


In [ ]:
# The context window must hold the longest prompt plus the whole generation budget.
PROMPT_TOKENS = 0
for ma, spec in TYPE_PROMPTS.items():
    longest = max(ALL_RECORDS[ma], key=lambda r: len(r['de_bai_mcq']))
    n_tok = len(tok(spec['template'].replace('{de_bai}', longest['de_bai_mcq'])).input_ids)
    print(f'{ma}: prompt {n_tok} tokens')
    PROMPT_TOKENS = max(PROMPT_TOKENS, n_tok)
MAX_MODEL_LEN = ((PROMPT_TOKENS + MAX_NEW_TOKENS + 1500) // 1024 + 1) * 1024
print(f'Longest prompt: {PROMPT_TOKENS} tokens -> MAX_MODEL_LEN={MAX_MODEL_LEN}')

N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'GPUs: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('Ready.')


In [ ]:
# =====================================================================
# BACKEND: the Calculator tool (SymPy with variable memory)
# =====================================================================
import re

import sympy as sp

ASSIGN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Names that already mean something to the Calculator (functions such as N, Min, floor and
# constants such as E, I, S). Storing a value under one of them would silently replace it,
# for example "N = 94" would make the later call N(...) fail for the rest of the question.
RESERVED_NAMES = {n for n in dir(sp) if not n.startswith('_')}
RELATIONAL = (sp.StrictLessThan, sp.LessThan, sp.StrictGreaterThan, sp.GreaterThan, sp.Unequality)


class Count(sp.Expr):
    """Count(condition, (n, low, high)): how many integers n from low to high make the condition true.

    The condition is decided exactly, one integer at a time (like a calculator table). It may itself
    contain another Count over a second variable; the inner Count is evaluated only after the outer
    variable has been replaced by a number, so nothing is ever expanded symbolically."""

    def __new__(cls, condition, limits, low=None, high=None):
        if low is None:
            var, low, high = limits
        else:
            var = limits
        obj = sp.Expr.__new__(cls, condition, var, sp.sympify(low), sp.sympify(high))
        if getattr(condition, 'free_symbols', set()) - {var}:
            return obj                      # still depends on another variable: wait
        return obj.doit()

    @property
    def free_symbols(self):
        condition, var, low, high = self.args
        return (condition.free_symbols - {var}) | low.free_symbols | high.free_symbols

    def doit(self, **hints):
        condition, var, low, high = self.args
        total = 0
        for value in range(int(low), int(high) + 1):
            decided = condition.subs(var, sp.Integer(value))
            decided = decided.doit() if hasattr(decided, 'doit') else decided
            if decided is sp.true or decided is True:
                total += 1
            elif not (decided is sp.false or decided is False):
                raise ValueError(f'the condition could not be decided for {var} = {value}: {decided}')
        return sp.Integer(total)


EXTRA_NAMES = {'Count': Count}
RESERVED_NAMES = RESERVED_NAMES | set(EXTRA_NAMES)


class Calculator:
    """SymPy calculator with variable memory (one fresh instance per question)."""

    def __init__(self, free_unknowns=()):
        self.free_unknowns = set(free_unknowns)
        # Free unknowns are registered up front as REAL symbols; letting
        # sympify create them lazily would make them generic complex symbols.
        self.ns = {name: sp.Symbol(name, real=True) for name in self.free_unknowns}

    def _undefined_names(self, expr):
        free = getattr(expr, 'free_symbols', set())
        missing = {str(s) for s in free} - self.free_unknowns
        if not missing:
            return None
        unknown = ', '.join(sorted(missing))
        in_memory = ', '.join(sorted(self.ns)) or '(nothing stored yet)'
        allowed = ', '.join(sorted(self.free_unknowns)) or '(none)'
        return (f'undefined name(s): {unknown}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {in_memory}. Declared free unknowns '
                f'(allowed without prior assignment): {allowed}. Either compute '
                'and store that name first (with "name = expression"), or '
                'rewrite the expression without it.')

    def _unknown_functions(self, parsed):
        """Names used as functions that the Calculator does not know (ceil, log10, int, ...).

        SymPy accepts any unknown name followed by parentheses and silently keeps it as an
        unevaluated symbol, so "count = ceil(115.36) - 1" would be stored as if it were a
        result. Refuse it with a message that says what to write instead."""
        items = parsed if isinstance(parsed, (list, tuple, sp.FiniteSet)) else [parsed]
        names = set()
        for item in items:
            for node in getattr(item, 'atoms', lambda *a: set())(sp.core.function.AppliedUndef):
                names.add(type(node).__name__)
        if not names:
            return None
        return (f'unknown function(s): {", ".join(sorted(names))}. The Calculator does not know '
                'these names, so nothing was computed. Use only: log, exp, sqrt, floor, '
                'ceiling, Abs, Min, Max, N, solve, expand, factor, diff, nroots, Eq. '
                'For example: round up with ceiling(u), round down with floor(u), a logarithm '
                'to base b as log(u)/log(b).')

    def _decide_relational(self, rel):
        """True or False for an inequality between constant expressions."""
        err = self._undefined_names(rel.lhs - rel.rhs)
        if err:
            return None, err
        if rel.lhs.free_symbols or rel.rhs.free_symbols:
            return None, ('cannot decide an inequality that still contains an '
                          'unknown. Substitute numbers first.')
        diff = sp.N(rel.lhs - rel.rhs, 50)
        if abs(diff) < sp.Float('1e-40'):
            diff = sp.Integer(0)
        table = {sp.StrictLessThan: diff < 0, sp.LessThan: diff <= 0,
                 sp.StrictGreaterThan: diff > 0, sp.GreaterThan: diff >= 0,
                 sp.Unequality: diff != 0}
        return ('True' if table[type(rel)] else 'False'), None

    def _boolean_result(self, name, truth):
        """Format a True/False answer; store it when it was assigned to a name."""
        text = 'True' if truth else 'False'
        if name:
            self.ns[name] = sp.true if truth else sp.false
            return f'{name} = {text}', None
        return text, None

    def evaluate(self, expression):
        """Return (result_text, error_text). Exactly one of them is None."""
        s = expression.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        name = None
        m = ASSIGN_RE.match(s)
        # "x = Eq(...)" is an assignment; a bare "Eq(a, b)" has no top level "=".
        if m and not s.lstrip().startswith('Eq('):
            name, s = m.group(1), m.group(2)
            if name in self.free_unknowns:
                return None, (f'"{name}" is a declared unknown used by solve(...) and cannot be '
                              'a variable name. Choose another name and store the value again.')
            if name in RESERVED_NAMES:
                return None, (f'"{name}" cannot be used as a variable name because it is the name '
                              'of a Calculator function or constant (for example N(expression, '
                              'digits) computes a decimal value). Choose another name, such as a '
                              'descriptive word, and store the value again.')

        try:
            parsed = sp.sympify(s, locals={**EXTRA_NAMES, **self.ns})
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        err = self._unknown_functions(parsed)
        if err:
            return None, err

        try:
            if isinstance(parsed, sp.Equality):
                err = self._undefined_names(parsed.lhs - parsed.rhs)
                if err:
                    return None, err
                same = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return self._boolean_result(name, same)

            if isinstance(parsed, RELATIONAL):
                text, err = self._decide_relational(parsed)
                if err:
                    return None, err
                return self._boolean_result(name, text == 'True')

            if isinstance(parsed, (sp.logic.boolalg.BooleanTrue, sp.logic.boolalg.BooleanFalse)):
                return self._boolean_result(name, bool(parsed))

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Result of solve(...): a list of solutions. Check each element
                # separately instead of simplifying the list as a whole.
                items = list(parsed)
                for item in items:
                    err = self._undefined_names(item)
                    if err:
                        return None, err
                if name:
                    self.ns[name] = items
                    return f'{name} = {items}', None
                return str(items), None

            value = sp.expand(sp.simplify(parsed))
            err = self._undefined_names(value)
            if err:
                return None, err
            if name:
                self.ns[name] = value
                return f'{name} = {value}', None
            return str(value), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# BATCHED ReAct LOOP (one function call = one problem type, all its questions):
# one vLLM call per round for every active question, then the Calculator
# runs separately for each question.
# =====================================================================
STOP_STR = 'Observation:'
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class QuestionState:
    """ReAct state of ONE question (own variable memory, own token budget)."""

    def __init__(self, rec, chat_prompt, free_unknowns):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.calculator = Calculator(free_unknowns=free_unknowns)
        # Forced prefix: the model must continue from "Thought:" right after
        # <think>, it cannot start with free prose.
        self.full_text = '<think>\nThought:'
        self.total_tokens = 0
        self.n_calls = 0
        self.tool_log = []
        self.done = False
        self.stop_reason = ''
        self.final_round = False          # tool limit reached: force the Final Answer


def extract_letter(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*(?:\\text\{\s*)?([A-D])\b',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


def run_type(ma_dang):
    spec = TYPE_PROMPTS[ma_dang]
    free_unknowns = set(spec['free_unknowns'])
    max_tool_calls = spec['max_tool_calls']
    records = ALL_RECORDS[ma_dang]
    if RUN_STT is not None:
        records = [r for r in records if r['STT'] in RUN_STT]
    out_path = f'/kaggle/working/{ma_dang.lower()}_react_calculator_full90.csv'
    print('=' * 72)
    print(f'{ma_dang}: {len(records)} questions')
    print('=' * 72)

    states = []
    for r in records:
        prompt_text = spec['template'].replace('{de_bai}', r['de_bai_mcq'])
        chat = tok.apply_chat_template([{'role': 'user', 'content': prompt_text}],
                                       tokenize=False, add_generation_prompt=True, enable_thinking=True)
        states.append(QuestionState(r, chat, free_unknowns))

    t0 = time.time()
    round_no = 0
    while True:
        active = [s for s in states if not s.done]
        if not active:
            break
        round_no += 1

        batch = []
        for s in active:
            model_input = s.chat_prompt + s.full_text
            room = MAX_MODEL_LEN - len(tok(model_input).input_ids) - 8
            n_new = min(MAX_NEW_TOKENS - s.total_tokens, room)
            if n_new <= 0:
                s.done = True
                s.stop_reason = 'out_of_context' if room <= 0 else 'out_of_token_budget'
                continue
            stop = None if s.final_round else [STOP_STR]
            batch.append((s, model_input, SamplingParams(
                temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                presence_penalty=PRESENCE_PENALTY, max_tokens=n_new,
                seed=SEED, stop=stop)))

        if not batch:
            break

        outs = llm.generate([p for _, p, _ in batch],
                            [sp_ for _, _, sp_ in batch], use_tqdm=False)

        calls_this_round = 0
        for (s, _, _), out in zip(batch, outs):
            o = out.outputs[0]
            s.full_text += o.text
            s.total_tokens += len(o.token_ids)

            if s.final_round:
                s.done = True
                s.stop_reason = 'tool_limit_reached'
                continue

            if o.stop_reason != STOP_STR:
                s.done = True
                s.stop_reason = ('model_finished' if o.finish_reason == 'stop'
                                 else 'out_of_tokens')
                continue

            s.n_calls += 1
            calls_this_round += 1
            match = None
            for mm in ACTION_INPUT_RE.finditer(o.text):
                match = mm
            expression = match.group(1).strip() if match else ''

            if not expression:
                observation = ('ERROR: no "Action Input:" line found. Write a Thought, '
                               'then "Action: Calculator", then "Action Input: <expression>".')
                s.tool_log.append(('(no Action Input)', observation))
            else:
                result, error = s.calculator.evaluate(expression)
                if error:
                    observation = (f'ERROR: {error}. Fix the syntax and try again '
                                   '(sqrt() for roots, ** for powers, log() for logarithms, '
                                   'Eq(a, b) to compare).')
                    s.tool_log.append((expression, f'ERROR: {error}'))
                else:
                    observation = result
                    s.tool_log.append((expression, result))

            s.full_text += f'{STOP_STR} {observation}\n'

            if s.n_calls >= max_tool_calls:
                s.full_text += ('\n[SYSTEM: tool-call limit reached, give your Final '
                                'Answer now, with no further Action.]\n')
                s.final_round = True

        print(f'  Round {round_no:2d}: {len(batch):2d} questions generating, '
              f'{calls_this_round:2d} tool calls, '
              f'{sum(1 for s in states if not s.done):2d} still active '
              f'({time.time()-t0:.0f}s)')

    print(f'Finished after {round_no} rounds, {(time.time()-t0)/60:.1f} minutes.')

    rows = []
    for s in states:
        chosen = extract_letter(s.full_text)
        n_err = sum(1 for _, obs in s.tool_log if str(obs).startswith('ERROR'))
        rows.append({
            'STT': s.rec['STT'],
            'ma_dang': s.rec.get('ma_dang', ma_dang),
            'dap_an_dung': s.rec['dap_an_letter'],
            'model_chon': chosen,
            'DUNG': chosen == s.rec['dap_an_letter'],
            'token': s.total_tokens,
            'tool_calls': s.n_calls,
            'syntax_errors': n_err,
            'stop_reason': s.stop_reason,
            'tool_log': ' | '.join(f'{ex} -> {obs}' for ex, obs in s.tool_log),
            'full_text': s.full_text,
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding='utf-8-sig')

    n_right = int(df['DUNG'].sum())
    n_none = int((df['model_chon'] == '').sum())
    print(f"RESULT {ma_dang}: {n_right} / {len(df)} correct ({df['DUNG'].mean()*100:.1f}%)   "
          f"wrong={len(df) - n_right - n_none}   undetermined={n_none}")
    print(f"Average tool calls per question : {df['tool_calls'].mean():.1f}   "
          f"Total syntax errors: {df['syntax_errors'].sum()} "
          f"({df['syntax_errors'].sum() / max(1, df['tool_calls'].sum()) * 100:.1f}%)   "
          f"Questions with NO tool call: {(df['tool_calls'] == 0).sum()}   "
          f"Average tokens: {df['token'].mean():.0f}")
    print(df['stop_reason'].value_counts().to_string())
    if (~df['DUNG']).any():
        print('--- WRONG OR UNDETERMINED QUESTIONS ---')
        print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'token',
                               'tool_calls', 'syntax_errors', 'stop_reason']].to_string(index=False))
    else:
        print(f'*** ALL {len(df)} QUESTIONS CORRECT ***')
    print('Saved:', out_path)
    return df


# =====================================================================
# RUN EVERY TYPE OF THIS NOTEBOOK, ONE AFTER THE OTHER
# =====================================================================
summary_rows = []
for ma in RUN_TYPES:
    d_ = run_type(ma)
    summary_rows.append({'ma_dang': ma, 'tong': len(d_), 'dung': int(d_['DUNG'].sum()),
                         'ti_le_dung': round(float(d_['DUNG'].mean()) * 100, 1),
                         'tool_calls_tb': round(float(d_['tool_calls'].mean()), 1),
                         'syntax_errors': int(d_['syntax_errors'].sum()),
                         'token_tb': round(float(d_['token'].mean()))})
    pd.DataFrame(summary_rows).to_csv(f'/kaggle/working/summary_{GROUP}.csv', index=False, encoding='utf-8-sig')

print()
print('=' * 72)
print('SUMMARY')
print('=' * 72)
print(pd.DataFrame(summary_rows).to_string(index=False))
